
# Course-End Project — Prove and Communicate ML Business Value
> **Solution Notebook** — one worked approach per activity (for review after your attempt).


In [ ]:

# ================================================
# Setup & Synthetic Data
# ================================================
import numpy as np, pandas as pd

rng = np.random.default_rng(123)

N_USERS = 80_000
REGIONS = ["NA","EU","LATAM","APAC"]
users = pd.DataFrame({
    "user_id": np.arange(N_USERS),
    "region": rng.choice(REGIONS, size=N_USERS, p=[0.35,0.25,0.25,0.15]),
    "device_id": rng.integers(10_000, 99_999, N_USERS)
})

# Generic events per user (supports both "good outcome" and "loss" framing)
tx_counts = rng.poisson(lam=1.6, size=N_USERS)
rows = []
start = pd.Timestamp("2025-03-01")

for uid, reg, dev, k in zip(users.user_id, users.region, users.device_id, tx_counts):
    if k == 0: 
        continue
    ts = start + pd.to_timedelta(rng.integers(0, 21, k), unit="D")
    base_good = {"NA":0.14, "EU":0.11, "LATAM":0.09, "APAC":0.10}[reg]     # e.g., conversion/retention
    base_loss = {"NA":0.007, "EU":0.005, "LATAM":0.012, "APAC":0.007}[reg] # e.g., fraud
    for t in ts:
        rows.append((uid, reg, dev, t, base_good, base_loss))

df = pd.DataFrame(rows, columns=["user_id","region","device_id","event_ts","p_good","p_loss"])
N = len(df)

# Arm assignment at user level by default (you can imagine geo designs later)
df["assigned_variation"] = rng.choice(["control","treatment"], size=N)

# Good outcome path (e.g., conversion or retention)
lift_rel = 0.03  # +3% relative
good_control = rng.random(N) < df["p_good"].values
good_treatment = rng.random(N) < (df["p_good"]*(1+lift_rel)).values
df["good_outcome"] = np.where(df["assigned_variation"]=="treatment", good_treatment, good_control)
df["arpu_or_margin"] = np.where(df["good_outcome"], rng.normal(8.5, 1.5, N).clip(3, 14), 0.0)

# Loss path (e.g., fraud); model flag & avoided loss
is_loss_true = rng.random(N) < df["p_loss"].values
df["is_loss_true"] = is_loss_true
df["loss_amount"] = np.where(is_loss_true, rng.lognormal(3.0, 0.8, N), 0.0)
df["model_flag"] = (df["assigned_variation"]=="treatment") & (rng.random(N) < 0.32)
df["loss_avoided"] = np.where(df["model_flag"] & is_loss_true, df["loss_amount"]*0.33, 0.0)

# Label delays & windows
label_delay_days = rng.lognormal(mean=2.0, sigma=0.9, size=N).astype(int)
df["label_available_ts"] = df["event_ts"] + pd.to_timedelta(label_delay_days, unit="D")

# Guardrails proxies
df["nps_delta"] = np.where(df["good_outcome"], rng.normal(0.01,0.02,N), rng.normal(0.0,0.02,N))
df["latency_ms"] = np.where(df["model_flag"], rng.normal(430, 60, N).clip(180, 800), rng.normal(310, 50, N).clip(180, 800))
df["support_tickets"] = rng.poisson(lam=np.where(df["model_flag"], 0.004, 0.003), size=N)

print("Events:", len(df), "Users:", N_USERS)
df.head()


In [ ]:

# ================================================
# Helper Functions
# ================================================
import numpy as np, pandas as pd
from math import ceil
from scipy.stats import norm

def metric_tree_template():
    return {
        "model_metrics": {"auc": None, "precision": None, "recall": None},
        "product_kpis": {"successful_actions": None, "net_retained_or_approved": None},
        "business_outcomes": {"incremental_revenue_or_savings": None},
        "costs": {"incentives": None, "infra": None, "ops_load": None},
        "guardrails": {"nps": None, "latency": None, "support_tickets": None}
    }

def window_labels(df, exposure_days=14, outcome_days=30, anchor=None):
    d = df.copy()
    anchor = pd.Timestamp("2025-03-01") if anchor is None else anchor
    exposure_end = anchor + pd.to_timedelta(exposure_days, unit="D")
    outcome_cutoff = exposure_end + pd.to_timedelta(outcome_days, unit="D")
    d = d[(d.event_ts >= anchor) & (d.event_ts < exposure_end)].copy()
    d["label_observed"] = d["label_available_ts"] <= outcome_cutoff
    return d, {"anchor": anchor, "exposure_end": exposure_end, "outcome_cutoff": outcome_cutoff}

def estimate_spillover(df):
    pivot = df.groupby(["device_id","assigned_variation"]).size().unstack(fill_value=0)
    cross_arm_devices = (pivot.gt(0).sum(axis=1) > 1).mean()
    dev_regs = df.groupby("device_id")["region"].nunique().mean()
    return {"cross_arm_device_share": float(cross_arm_devices),
            "avg_regions_per_device": float(dev_regs),
            "spillover_risk_score": float(0.5*cross_arm_devices + 0.5*min(dev_regs/3,1.0))}

def n_per_arm_for_proportions(p_ctrl, mde_abs, alpha=0.05, power=0.8):
    z_alpha = norm.ppf(1 - alpha/2); z_beta = norm.ppf(power)
    p_treat = max(1e-9, min(1-1e-9, p_ctrl - mde_abs))
    var = p_ctrl*(1-p_ctrl) + p_treat*(1-p_treat)
    return int(np.ceil(var * (z_alpha + z_beta)**2 / (mde_abs**2)))

def design_effect(cluster_size, icc):
    return 1 + (cluster_size - 1) * icc

def cuped_adjust(y_post, y_pre):
    cov = np.cov(y_post, y_pre, ddof=1); theta = 0.0 if cov[1,1]==0 else cov[0,1]/cov[1,1]
    y_adj = y_post - theta * y_pre
    return y_adj, theta

def lift_to_revenue(baseline_rate, lift, population, adoption, margin_per_event):
    treated = population * adoption
    base_events = baseline_rate * treated
    treatment_events = base_events * (1 + lift)
    delta_events = treatment_events - base_events
    delta_revenue = delta_events * margin_per_event
    return float(delta_revenue), float(delta_events)

def total_costs(incentive_per_event, delta_events, infra, labeling, retraining):
    variable = incentive_per_event * max(delta_events, 0.0)
    total = variable + infra + labeling + retraining
    return float(total), float(variable)

def payback_period(monthly_net_cashflow, upfront, max_months=36):
    if monthly_net_cashflow <= 0: return None
    return int(np.ceil(upfront / monthly_net_cashflow))

def npv_of_stream(monthly_net, months, monthly_discount, upfront=0.0):
    disc = sum([monthly_net / ((1+monthly_discount)**t) for t in range(1, months+1)])
    return float(disc - upfront)

def guardrail_report(df_obs, thresholds=None):
    if thresholds is None:
        thresholds = {"nps_min_delta":0.0,"latency_max_ms":500,"tickets_diff_max":0.001}
    d = df_obs.copy()
    treat = d[d.assigned_variation=="treatment"]; ctrl = d[d.assigned_variation=="control"]
    nps_diff = treat.nps_delta.mean() - ctrl.nps_delta.mean()
    latency_diff = treat.latency_ms.mean() - ctrl.latency_ms.mean()
    tickets_diff = treat.support_tickets.mean() - ctrl.support_tickets.mean()
    checks = {"NPS Δ >= 0": nps_diff >= thresholds["nps_min_delta"],
              "Latency within limit": latency_diff <= thresholds["latency_max_ms"],
              "Tickets not higher": tickets_diff <= thresholds["tickets_diff_max"]}
    decision = "GO" if all(checks.values()) else "NO-GO"
    return {"nps_diff": float(nps_diff), "latency_diff_ms": float(latency_diff),
            "tickets_diff": float(tickets_diff), "checks": checks, "decision": decision}

def bootstrap_profit_ci(base_rate, lift, population, adoption, margin, incent, infra, labeling, retraining, n_boot=300, seed=9):
    rng = np.random.default_rng(seed)
    treated = int(population * adoption)
    base_events = rng.binomial(treated, base_rate, size=n_boot)
    treat_events = rng.binomial(treated, min(1.0, base_rate*(1+lift)), size=n_boot)
    delta_events = (treat_events - base_events).astype(float)
    margins = rng.normal(margin, 0.8, size=n_boot).clip(0, None)
    delta_revenue = delta_events * margins
    var_cost = np.maximum(delta_events, 0.0) * incent
    profit = delta_revenue - (var_cost + infra + labeling + retraining)
    lo, hi = np.percentile(profit, [2.5, 97.5])
    return float(profit.mean()), float(lo), float(hi)

print("Helpers ready.")



## Activity 1 — Metric Tree & Context Choice (LO1)
Choose a context (churn-save / fraud / recommender) and complete the metric tree.


In [ ]:

# SOLUTION — Activity 1
context = "recommender"
metric_tree = {
    "model_metrics": {
        "auc": "Improves ranking quality at top of list",
        "precision": "Reduces wasted impressions on low-propensity items",
        "recall": "Covers more high-propensity users/items"
    },
    "product_kpis": {
        "successful_actions": "Clicks-to-add-to-cart and checkout starts",
        "net_retained_or_approved": "Incremental orders net of any induced churn"
    },
    "business_outcomes": {
        "incremental_revenue_or_savings": "Δorders × gross margin per order"
    },
    "costs": {
        "incentives": "Promo costs for nudges",
        "infra": "Serving + storage + labeling",
        "ops_load": "Support/agent time for escalations"
    },
    "guardrails": {
        "nps": "Do not degrade NPS",
        "latency": "Keep P95 latency <= 500 ms",
        "support_tickets": "No surge in tickets"
    }
}
metric_tree



## Activity 2 — Measurement Plan (LO1)
Primary metric, counterfactual, windows, and guardrails with thresholds.


In [ ]:

# SOLUTION — Activity 2
primary_metric = "Incremental revenue (good outcome path)"
counterfactual = "BAU"
exposure_days = 14; outcome_days = 28
df_obs, meta = window_labels(df, exposure_days, outcome_days)
{"primary_metric": primary_metric, "counterfactual": counterfactual, "windows": (exposure_days, outcome_days)}



## Activity 3 — Experiment/Quasi-Experiment Design (LO2)
Unit of assignment, design choice, and power sketch.


In [ ]:

# SOLUTION — Activity 3
unit = "user"; design = "user_ab"
baseline_rate = (df_obs.good_outcome).mean()
target_mde = 0.005
n_naive = n_per_arm_for_proportions(baseline_rate, target_mde)
cluster_size = 1500; icc = 0.02
deff = design_effect(cluster_size, icc); n_cluster = int(np.ceil(n_naive*deff))
{"n_naive": n_naive, "design_effect": float(deff), "n_clustered": n_cluster}



## Activity 4 — Lift → Financials (LO3)
Compute incremental profit, payback, and 12-month NPV. Then run sensitivity.


In [ ]:

# SOLUTION — Activity 4
import matplotlib.pyplot as plt
treat = df_obs[df_obs.assigned_variation=="treatment"]; ctrl  = df_obs[df_obs.assigned_variation=="control"]
lift_rel = (treat.good_outcome.mean() / max(ctrl.good_outcome.mean(),1e-9)) - 1

population = 200_000; adoption = 0.6; margin_per_event = 8.0
incentive_cost_per_event = 0.80; infra_cost = 6000; labeling_cost = 2500; retraining_cost = 1500
upfront = 30_000; discount_rate_monthly = 0.12 ** (1/12)

delta_revenue, delta_events = lift_to_revenue(ctrl.good_outcome.mean(), lift_rel, population, adoption, margin_per_event)
total_cost, var_cost = total_costs(incentive_cost_per_event, delta_events, infra_cost, labeling_cost, retraining_cost)
incremental_profit = delta_revenue - total_cost
payback = payback_period(incremental_profit, upfront, max_months=36)
npv_12m = npv_of_stream(incremental_profit, 12, discount_rate_monthly, upfront)

roi_table = pd.DataFrame([{"delta_revenue":delta_revenue,"variable_cost":var_cost,"fixed_costs":total_cost - var_cost,"incremental_profit":incremental_profit,"payback_months":payback,"npv_12m":npv_12m}])
roi_table


In [ ]:

# Sensitivity (±20%); tornado plot
ranges = {"treatment_lift":0.20,"gross_margin_per_event":0.20,"incentive_cost_per_event":0.20,"infra_cost_per_period":0.20,"adoption":0.20}
inputs = {"baseline_rate": float(ctrl.good_outcome.mean()), "treatment_lift": float(lift_rel), "population": 200_000, "adoption":0.6,
          "gross_margin_per_event": 8.0, "incentive_cost_per_event":0.80, "infra_cost_per_period":6000,
          "labeling_cost_per_period":2500, "retraining_cost_per_period":1500, "upfront_investment":30000, "discount_rate_monthly": float(0.12 ** (1/12))}
def sensitivity_scan(base_inputs, ranges, months=12):
    out = []
    for k, pct in ranges.items():
        for direction in (-1, +1):
            t = dict(base_inputs); t[k] = base_inputs[k]*(1 + direction*pct)
            dr, de = lift_to_revenue(t["baseline_rate"], t["treatment_lift"], t["population"], t["adoption"], t["gross_margin_per_event"])
            tc, vc = total_costs(t["incentive_cost_per_event"], de, t["infra_cost_per_period"], t["labeling_cost_per_period"], t["retraining_cost_per_period"])
            ip = dr - tc
            disc = sum([ip / ((1+t["discount_rate_monthly"])**m) for m in range(1,13)]) - t["upfront_investment"]
            out.append({"driver":k,"direction":"minus" if direction==-1 else "plus","NPV": float(disc)})
    df_s = pd.DataFrame(out).pivot(index="driver", columns="direction", values="NPV")
    df_s["swing"] = (df_s["plus"] - df_s["minus"]).abs()
    return df_s.sort_values("swing", ascending=False).reset_index()

swings = sensitivity_scan(inputs, ranges, months=12)
swings


In [ ]:

import matplotlib.pyplot as plt
plt.figure()
plt.barh(swings["driver"], swings["swing"])
plt.xlabel("NPV swing (absolute)"); plt.ylabel("Driver"); plt.title("Sensitivity Tornado — NPV Swing by Driver (±20%)")
plt.show()



## Activity 5 — Impact Dashboard & Guardrails (LO3)
Plot trends and compute a GO/NO-GO.


In [ ]:

# SOLUTION — Activity 5
import matplotlib.pyplot as plt
weeks = pd.date_range("2025-03-03", periods=8, freq="W-MON")
primary = pd.Series(np.linspace(max(0.0, lift_rel*0.8), lift_rel*1.1, 8), index=weeks)
plt.figure(); plt.plot(primary.index, primary.values); plt.xlabel("Week"); plt.ylabel("Primary metric (lift proxy)"); plt.title("Weekly Primary Metric Trend"); plt.show()

nps = pd.Series(np.linspace(0.0, 0.02, 8), index=primary.index)
latency = pd.Series(np.linspace(470, 510, 8), index=primary.index)
tickets = pd.Series(np.linspace(10, 14, 8), index=primary.index)

plt.figure(); plt.plot(nps.index, nps.values); plt.xlabel("Week"); plt.ylabel("NPS delta"); plt.title("NPS Delta (weekly)"); plt.show()
plt.figure(); plt.plot(latency.index, latency.values); plt.xlabel("Week"); plt.ylabel("Latency (ms)"); plt.title("Approval Latency (weekly)"); plt.show()
plt.figure(); plt.plot(tickets.index, tickets.values); plt.xlabel("Week"); plt.ylabel("Support tickets"); plt.title("Support Tickets (weekly)"); plt.show()

thresholds = {"roi_min":0.0,"payback_max_months":6,"nps_min_delta":0.0,"latency_max_ms":500,"tickets_max_increase":2}
roi_ok = (roi_table["incremental_profit"].iloc[0] >= 0)
payback_ok = (roi_table["payback_months"].iloc[0] is not None) and (roi_table["payback_months"].iloc[0] <= thresholds["payback_max_months"])
nps_ok = (nps.iloc[-1] - nps.iloc[0]) >= thresholds["nps_min_delta"]
latency_ok = (latency.iloc[-1] <= thresholds["latency_max_ms"])
tickets_ok = (tickets.iloc[-1] - tickets.iloc[0]) <= thresholds["tickets_max_increase"]
decision = "GO" if all([roi_ok, payback_ok, nps_ok, latency_ok, tickets_ok]) else "NO-GO"
pd.DataFrame([{"decision": decision, "ROI >= 0": roi_ok, "Payback <= 6 mo": payback_ok, "NPS Δ >= 0": nps_ok, "Latency <= 500 ms": latency_ok, "Tickets +Δ ≤ 2": tickets_ok}])



## Activity 6 — Exec Summary & Recommendation (LO3)
Write an 8–10 line executive summary with a clear GO/NO-GO and staged rollout.



**Executive Summary (example):**  
Our recommender increased the good-outcome rate by ~3% (user A/B, 28-day window). Primary metric: Incremental revenue with BAU counterfactual. ROI: +$X/month incremental profit, payback ~Y months, 12‑month NPV ~$Zk; sensitivity shows NPV most sensitive to adoption and lift. Guardrails are green (NPS flat to +0.5pt; latency P95 ≤ 500 ms; tickets stable). Risks: spillover minimal; label delays handled with 28d window; follow-up geo validation planned. **Decision: GO** with staged rollout (10%→50%→100%) and rollback on guardrail fail or CI crossing zero. Next: add regional drill-downs and automate post-launch drift/CI monitoring.


> End of Solution.